# Grid UCI BNN — Sampler Investigation

Loads **all available samplers** for one `(dataset, split)` and compares them across:
metrics, sparsity, calibration, posterior noise, ESS, and predictive intervals.

Adapted from `uci_investigate.ipynb` for the `gpu_friendly` tree: `grid_boomerang` / `grid_sticky_boomerang` / `nuts` / `nuts_horseshoe` instead of zigzag/boomerang x sticky x PLI, and noise is learned by every sampler here (no fixed-noise variant, so the old notebook's `LEARNED_NOISE` filter and `target_cache` keyed by bool are gone).

In [ ]:
from __future__ import annotations

import math
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import torch
from torch import Tensor
from torch.distributions import Normal

plt.rcParams.update({
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.grid":          True,
    "grid.alpha":         0.3,
    "font.size":          11,
})

import os
if Path.cwd().name == "notebooks":
    os.chdir("..")

## 1. Choose (dataset, split)

In [ ]:
# Sections 1-6 work on RESAMPLED draws (produced by uci_bnn_grid.py, which
# calls resample_*_path_torch on the full skeleton). Section 7 works on RAW
# SKELETON files (produced by uci_bnn_grid_skeleton.py, no resampling) --
# see its own RESULTS_DIR/SKELETON_DIR further down. These are two
# different output trees; do not point this RESULTS_DIR at the skeleton
# directory, sections 1-6 will KeyError on missing "samples"/"x_ref".
RESULTS_DIR = Path("results/paper/deep_wide")

DATASET  = "boston"
SPLIT_ID = 0

# Display-name overrides (stem -> label)
LABELS = {
    "grid_zigzag":           "Grid ZigZag",
    "grid_sticky_zigzag":    "Grid Sticky ZigZag",
    "grid_boomerang":        "Grid Boomerang",
    "grid_sticky_boomerang": "Grid Sticky Boomerang",
    "nuts":                  "NUTS",
    "nuts_horseshoe":        "NUTS-HS",
    "tf_boomerang":          "TF Boome",
}

# Colour palette -- one colour per base sampler family
COLORS = {
    "grid_zigzag":           "#15A6DB",
    "grid_sticky_zigzag":    "#15A6DB",
    "grid_boomerang":        "#FA0000",
    "grid_sticky_boomerang": "#FA0000",
    "nuts":                  "#0CCA38",
    "nuts_horseshoe":        "#0CCA38",
    "tf_boomerang":          "#CE0577",
}

# Linestyle: solid for sticky/HS, dashed for vanilla
LINESTYLES = {
    "grid_zigzag":           "--",
    "grid_sticky_zigzag":    "-",
    "grid_boomerang":        "--",
    "grid_sticky_boomerang": "-",
    "nuts":                  "--",
    "nuts_horseshoe":        "-",
    "tf_boomerang":          "-",
}

## 2. Load runs and reconstruct data split

In [ ]:
from sazz.gpu_friendly.scripts.uci_bnn_grid import (
    load_raw_datasets, make_split, build_target, BNNConfig, BASE_SEED, DTYPE, DEVICE,
)

split_dir = RESULTS_DIR / DATASET / f"split_{SPLIT_ID:02d}"
pt_files  = sorted(split_dir.glob("*.pt")) if split_dir.exists() else []

print(split_dir)
if not pt_files:
    print(f"  No resampled runs found here yet -- sections 1-6 (metrics/RMSE/NLL/ESS/"
          f"sparsity/noise) need output from uci_bnn_grid.py (NOT uci_bnn_grid_skeleton.py) "
          f"and will skip themselves below. Jump to section 7 for skeleton-only dynamics, "
          f"which already works from the .pt files you have.")
else:
    print(f"Found {len(pt_files)} run(s) in {split_dir}:")
    for p in pt_files:
        print(f"  {p.stem}")

In [ ]:
data = X_test = y_test = y_std = None

if pt_files:
    print("Loading raw dataset for test-set reconstruction...")
    raw = load_raw_datasets((DATASET,))
    X_all, y_all = raw[DATASET]

    data   = make_split(X_all, y_all, seed=BASE_SEED + SPLIT_ID, dtype=DTYPE, device=DEVICE)
    X_test = data["X_test"]
    y_test = data["y_test"]
    y_std  = data["y_std"]
    print(f"  X_test: {X_test.shape}   y_std: {y_std:.4f}")
else:
    print("Skipping -- no resampled runs to build a test set for (see previous cell).")

## 3. Rebuild target and compute predictions

One `BayesianModule` is built (shared by every run in this split -- they all share the same architecture/config, mirroring `uci_bnn_grid.py`'s `run_split`). Predictions go through `bm.module`/`bm.param_dict_fn` via `functional_call`, replacing the old tree's `TorchTarget`/`ModuleGaussianLikelihood.predict()`.

In [ ]:
bm = None

@torch.no_grad()
def predict_all(bm, weight_samples: Tensor, X_new: Tensor) -> Tensor:
    return torch.stack([
        torch.func.functional_call(bm.module, bm.param_dict_fn(beta), (X_new,)).squeeze(-1)
        for beta in weight_samples
    ])  # [S, N]


runs: dict[str, dict] = {}   # stem -> payload

if not pt_files:
    print("Skipping -- no resampled runs (see cell above). runs = {}")
else:
    for pt in pt_files:
        stem = pt.stem
        run  = torch.load(pt, map_location="cpu", weights_only=False)

        if bm is None:
            cfg = BNNConfig(
                layer_sizes=run["layer_sizes"],
                activation=run["activation"],
                prior_sigma_scale=run["prior_sigma_scale"],
            )
            print(f"  Building target (layer_sizes={run['layer_sizes']}, act={run['activation']})...")
            bm, _, _ = build_target(data, cfg)
            print(f"    D = {bm.D}")

        samples = run["samples"].to(dtype=DTYPE)   # [S, D], last column is log_sigma
        weight_samples = samples[:, :-1]
        
        if samples.shape[0] == 0:
            print(f"SKIP: 0 samples ")
            continue

        preds     = predict_all(bm, weight_samples, X_test)   # [S, N]
        mean_pred = preds.mean(0)
        epist_std = preds.std(0)

        noise_samples = samples[:, -1].exp()       # [S]
        noise_std_eff = float(noise_samples.mean())

        total_std = (epist_std ** 2 + noise_std_eff ** 2).sqrt()

        runs[stem] = dict(
            run            = run,
            samples        = samples,
            weight_samples = weight_samples,
            mean_pred      = mean_pred,
            epist_std      = epist_std,
            total_std      = total_std,
            noise_std_eff  = noise_std_eff,
            noise_samples  = noise_samples,
            label          = LABELS.get(stem, stem),
            color          = COLORS.get(stem, "grey"),
            ls             = LINESTYLES.get(stem, "-"),
        )
        print(f"  [{stem}] done — {samples.shape[0]} samples")

sampler_order = list(runs.keys())

In [ ]:
# MAP prediction from x_ref (Adam estimate) -- one evaluation per run to verify x_ref consistency
print("MAP predictions (x_ref):")
for stem, s in runs.items():
    x_ref = s["run"]["x_ref"]
    if x_ref is None:
        print(f"  {s['label']}: no x_ref stored")
        continue
    x_ref = x_ref.to(dtype=DTYPE)
    weights_ref = x_ref[:-1]
    with torch.no_grad():
        pred = torch.func.functional_call(bm.module, bm.param_dict_fn(weights_ref), (X_test,)).squeeze(-1)
    noise = float(x_ref[-1].exp())
    rmse  = float(((pred - y_test) ** 2).mean().sqrt()) * y_std
    print(f"  {s['label']}: RMSE={rmse:.3f}  noise_std={noise:.3f}")

In [ ]:
for stem, s in runs.items():
    x_ref = s["run"]["x_ref"]
    if x_ref is None:
        continue
    x_ref = x_ref.to(dtype=DTYPE)
    ws = s["samples"].to(dtype=DTYPE)
    dists = (ws - x_ref).norm(dim=1)
    print(f"{s['label']}: mean dist={dists.mean():.3f}  std={dists.std():.3f}  max={dists.max():.3f}")

In [ ]:
# if not runs:
#     print("Skipping -- no resampled runs loaded (see section 2).")
# else:
#     _, x_ref_bm, Sigma_inv = build_target(data, BNNConfig(
#         layer_sizes=list(runs.values())[0]["run"]["layer_sizes"],
#         activation=list(runs.values())[0]["run"]["activation"],
#         prior_sigma_scale=list(runs.values())[0]["run"]["prior_sigma_scale"],
#     ))
#     si = Sigma_inv  # 1-D diagonal vector [D]

#     print(f"Shape : {si.shape}")
#     print(f"Min   : {si.min().item():.4g}")
#     print(f"Max   : {si.max().item():.4g}")
#     print(f"Ratio : {(si.max() / si.min()).item():.4g}")
#     print(f"Mean  : {si.mean().item():.4g}")
#     print(f"Median: {si.median().item():.4g}")

#     # Distribution of implied std devs (orbit radii per coordinate)
#     sigma_diag = (1.0 / si).sqrt()
#     print(f"\nImplied per-coord std (Sigma_sqrt diagonal):")
#     print(f"  Min   : {sigma_diag.min().item():.4g}")
#     print(f"  Max   : {sigma_diag.max().item():.4g}")
#     print(f"  Mean  : {sigma_diag.mean().item():.4g}")
#     print(f"  Median: {sigma_diag.median().item():.4g}")

In [ ]:
# if not runs:
#     print("Skipping -- no resampled runs loaded (see section 2).")
# else:
#     # Evaluate energy at x_ref, then take a few more Adam steps to sanity-check
#     # the MAP is actually near-converged.
#     e_ref = float(bm.energy(x_ref_bm))

#     beta = x_ref_bm.clone().requires_grad_(True)
#     opt = torch.optim.Adam([beta], lr=1e-3)
#     energies = []
#     for i in range(2000):
#         opt.zero_grad()
#         loss = bm.energy(beta)
#         loss.backward()
#         opt.step()
#         if i % 200 == 0:
#             energies.append(loss.item())

#     print(f"Energy at x_ref      : {e_ref:.4f}")
#     print(f"Energy after 2k more : {energies[-1]:.4f}")
#     print(f"Drop                 : {e_ref - energies[-1]:.4f}")
#     print(energies)

## 4. Metrics table

In [ ]:
from sazz.utils.metrics import ess_per_coord
import pandas as pd

def compute_rmse(y_true, mean_pred, y_std):
    return float(((mean_pred - y_true) ** 2).mean().sqrt()) * y_std

def compute_nll(y_true, mean_pred, total_std, y_std):
    ll = (-0.5 * ((y_true - mean_pred) / total_std) ** 2
          - total_std.log() - 0.5 * math.log(2 * math.pi)).mean()
    return float(-ll + math.log(y_std))

def compute_crps(y_true, mean_pred, total_std, y_std):
    d = Normal(0.0, 1.0)
    sigma = total_std * y_std
    z = (y_true * y_std - mean_pred * y_std) / sigma
    return float((sigma * (z * (2*d.cdf(z) - 1) + 2*d.log_prob(z).exp()
                           - 1/math.sqrt(math.pi))).mean())

def compute_coverage(y_true, mean_pred, total_std, level=0.9):
    z = Normal(0.0, 1.0).icdf(torch.tensor(0.5 + level / 2))
    return float(((y_true - mean_pred).abs() <= z * total_std).float().mean())

metrics_df = None

if not runs:
    print("Skipping -- no resampled runs loaded (see section 2).")
else:
    rows = []
    for stem, s in runs.items():
        #ess         = ess_per_coord(s["weight_samples"])
        elapsed     = s["run"]["elapsed_sec"]
        grad_evals  = s["run"].get("gradient_evals")  # NUTS (num_steps) or PDMP; None if not tracked
        rows.append({
            "Sampler":    s["label"],
            "RMSE":       compute_rmse(y_test, s["mean_pred"], y_std),
            "NLL":        compute_nll(y_test, s["mean_pred"], s["total_std"], y_std),
            "CRPS":       compute_crps(y_test, s["mean_pred"], s["total_std"], y_std),
            "Cov 90%":    compute_coverage(y_test, s["mean_pred"], s["total_std"], 0.90),
            "Cov 95%":    compute_coverage(y_test, s["mean_pred"], s["total_std"], 0.95),
           # "ESS min":    float(ess.min()),
            #"ESS/s":      float(ess.min()) / elapsed,
            # ESS/gradient-eval: the fairest cross-family currency here -- PDMP
            # "iterations" and NUTS "draws" aren't comparable units, but both
            # samplers pay in gradient evaluations of the (log-)target.
            #"ESS/grad":   (float(ess.min()) / grad_evals) if grad_evals else float("nan"),
            "Grad evals": grad_evals if grad_evals is not None else float("nan"),
            "Time (s)":   elapsed,
        })

    metrics_df = pd.DataFrame(rows).set_index("Sampler")

def highlight_coverage(col):
    target = 0.90 if "90" in col.name else 0.95
    best = (col - target).abs().idxmin()
    return ["background-color: #68dc0f" if idx == best else "" for idx in col.index]

In [ ]:
if metrics_df is None:
    print("Skipping -- no resampled runs loaded (see section 2).")
else:
    display(
        metrics_df.style
        .highlight_min(subset=["RMSE", "NLL", "CRPS"], color="#68dc0f")
        .apply(highlight_coverage, subset=["Cov 90%", "Cov 95%"])
        .format(precision=5, na_rep="n/a")
    )

## 4b. Sampler cost diagnostics (all samplers)

Compute-cost view across all four samplers, in a common currency (gradient evaluations of the log-target), since PDMP "skeleton events" and NUTS "draws" aren't directly comparable units -- both pay per gradient evaluation, so `Grad evals/s` and `ESS/grad` (section 4) are the fairest cross-family comparisons here. `t_max` columns are grid-sampler-only (NUTS has no adaptive horizon). Fields are only present in `.pt` files saved after `save_run`/NUTS runners started persisting them -- older runs show `n/a`; re-run the relevant sampler(s) to backfill.

In [ ]:
diag_rows = []
for stem, s in runs.items():
    r = s["run"]

    n_events   = r.get("n_events")
    elapsed    = r.get("elapsed_sec")
    grad_evals = r.get("gradient_evals")
    t_max_log  = r.get("grid_t_max_log")
    bound_viol = r.get("bound_violations")

    diag_rows.append({
        "Sampler":          s["label"],
        "Events/s":         (n_events / elapsed) if (n_events and elapsed) else float("nan"),
        "Grad evals/s":     (grad_evals / elapsed) if (grad_evals and elapsed) else float("nan"),
        "Grad evals/event": (grad_evals / n_events) if (grad_evals and n_events) else float("nan"),
        "t_max mean":       float(np.mean(t_max_log)) if t_max_log else float("nan"),
        "t_max min":        float(np.min(t_max_log)) if t_max_log else float("nan"),
        "t_max max":        float(np.max(t_max_log)) if t_max_log else float("nan"),
        "Bound violations": bound_viol if bound_viol is not None else float("nan"),
        "Time (s)":         elapsed if elapsed is not None else float("nan"),
    })

grid_diag_df = pd.DataFrame(diag_rows).set_index("Sampler")
grid_diag_df.style.format(precision=4, na_rep="n/a")

## 5. Sparsity profiles

Meaningful for `grid_sticky_boomerang`; the other samplers won't show real sparsity but are plotted for comparison.

In [ ]:
if not runs:
    print("Skipping -- no resampled runs loaded (see section 2).")
else:
    FREEZE_THR = 1e-3
    window = 100

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    for stem, s in runs.items():
        ws = s["weight_samples"]  # [S, D]

        # --- Sorted sparsity scatter ---
        sparsity = (ws.abs() < FREEZE_THR).float().mean(dim=0).numpy()
        sorted_sparsity = np.sort(sparsity)
        axes[0].scatter(range(len(sorted_sparsity)), sorted_sparsity,
                        alpha=0.5, color=s["color"], label=s["label"], s=8,
                        marker=("o" if s["ls"] == "-" else "x"))

        # --- Model size over samples ---
        model_size = (ws.abs() >= FREEZE_THR).sum(dim=1).float()
        rolling_mean = model_size.unfold(0, window, 1).mean(dim=1)
        axes[1].plot(model_size.numpy(), alpha=0.1, color=s["color"])
        axes[1].plot(range(window // 2, len(rolling_mean) + window // 2),
                     rolling_mean.numpy(), color=s["color"], ls=s["ls"],
                     lw=1.6, label=s["label"])

        print(f"{s['label']}: sparsity={sparsity.mean():.3f}, "
              f"mean active params={model_size.mean():.1f} / {ws.shape[1]}")

    axes[0].set_xlabel("Parameter rank (sorted by sparsity)")
    axes[0].set_ylabel("P(|param| < threshold)")
    axes[0].set_title(f"{DATASET.capitalize()} — per-parameter sparsity (sorted)")
    axes[0].legend(fontsize=8)

    axes[1].set_xlabel("Sample")
    axes[1].set_ylabel("# active params")
    axes[1].set_title(f"{DATASET.capitalize()} — model size over samples")
    axes[1].legend(fontsize=8)

    plt.tight_layout()
    plt.show()

## 6. Posterior noise

Every run learns noise, so this section always has data (no `if ln_runs:` guard needed, unlike the old notebook).

In [ ]:
import numpy as np
fig, ax = plt.subplots(figsize=(7, 4))

# common range from robust percentiles across all runs
allns = np.concatenate([s["noise_samples"].numpy() for s in runs.values()])
allns = allns[np.isfinite(allns)]
lo, hi = np.percentile(allns, [0.0001, 99.9999])

for stem, s in runs.items():
    ns = s["noise_samples"].numpy()
    ns = ns[np.isfinite(ns)]
    ax.hist(ns, bins=60, range=(lo, hi), density=True, alpha=0.4,
            color=s["color"], histtype="stepfilled", edgecolor="none")
    ax.axvline(ns.mean(), color=s["color"], lw=1.8, ls=s["ls"],
               label=f"{s['label']} (mean={ns.mean():.3f}, "
                     f"[{ns.min():.3f}, {ns.max():.3f}])")
ax.set_xlim(lo, hi)
ax.set_xlabel(r"$\sigma$ (standardised scale)")
ax.set_ylabel("Density")
ax.set_title(f"{DATASET.capitalize()} — posterior noise")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()


In [ ]:
runs['grid_sticky_zigzag']['run']

In [ ]:
# --- Posterior inclusion probability for one (PIW, sampler), sliced onto the network ---
import matplotlib as mpl

STEM     = "grid_sticky_zigzag"   # "grid_sticky_zigzag" or "grid_sticky_boomerang"
ZERO_TOL = 0.0                     # exact-zero test; use e.g. 1e-8 for near-zero

r = runs[STEM]

# weight_samples is [S, D-1] -- network weights + biases only, log_sigma already dropped
ws_arr    = r["weight_samples"].numpy()
S, Dw     = ws_arr.shape
incl_prob = (np.abs(ws_arr) > ZERO_TOL).mean(axis=0)          # P(active), one per network coord

layer_sizes = runs[STEM]['run']["layer_sizes"]                          # e.g. [13, 50, 1]
activation  = runs[STEM]['run']["activation"]
n_net = sum(n_in * n_out + n_out for n_in, n_out in zip(layer_sizes[:-1], layer_sizes[1:]))
assert n_net == Dw, (n_net, Dw)                                # weights+biases, no log_sigma

# Walk named_parameters() order: layers.i.weight [n_out, n_in], then layers.i.bias [n_out]
W_incl, b_incl, off = [], [], 0
for n_in, n_out in zip(layer_sizes[:-1], layer_sizes[1:]):
    W_incl.append(incl_prob[off:off + n_out * n_in].reshape(n_out, n_in)); off += n_out * n_in
    b_incl.append(incl_prob[off:off + n_out]);                            off += n_out
assert off == Dw, (off, Dw)

print(f"{LABELS[STEM]} (S={S} draws)")
for li, (Wp, bp) in enumerate(zip(W_incl, b_incl)):
    print(f"  layer {li}: W{Wp.shape}  mean incl={Wp.mean():.3f}   "
          f"bias mean incl={bp.mean():.3f}   fully-excluded weights={(Wp == 0).mean():.3f}")


In [ ]:
# --- Draw the network, edges coloured by posterior inclusion probability ---
cmap = mpl.colormaps["RdYlGn"]                  # 0 -> red (excluded), 1 -> green (included)
norm = mpl.colors.Normalize(vmin=0.0, vmax=1.0)

xs = np.arange(len(layer_sizes))
def _ys(n):
    return np.linspace(0, 1, n) if n > 1 else np.array([0.5])
node_y = [_ys(n) for n in layer_sizes]

fig, ax = plt.subplots(figsize=(4 + 1.6 * len(layer_sizes), 9))

# edges: weight[j, k] connects node k in layer li to node j in layer li+1
for li, Wp in enumerate(W_incl):
    n_out, n_in = Wp.shape
    y0, y1 = node_y[li], node_y[li + 1]
    order = np.argsort(Wp.ravel())             # draw excluded (red) first, included (green) on top
    for idx in order:
        j, k = divmod(idx, n_in)
        p = Wp[j, k]
        ax.plot([xs[li], xs[li + 1]], [y0[k], y1[j]],
                color=cmap(norm(p)), lw=0.4 + 2.2 * p,
                alpha=0.15 + 0.85 * p, zorder=1, solid_capstyle="round")

# nodes coloured by mean incoming-weight inclusion (input layer: neutral grey)
for li, n in enumerate(layer_sizes):
    node_c = ["0.6"] * n if li == 0 else [cmap(norm(W_incl[li - 1][j].mean())) for j in range(n)]
    ax.scatter(np.full(n, xs[li]), node_y[li], s=260, c=node_c,
               edgecolors="black", linewidths=0.8, zorder=3)

labels = ([f"input\n({layer_sizes[0]})"]
          + [f"hidden {i}\n({s}, {activation})" for i, s in enumerate(layer_sizes[1:-1], 1)]
          + [f"output\n({layer_sizes[-1]})"])
for x, lab in zip(xs, labels):
    ax.text(x, -0.08, lab, ha="center", va="top", fontsize=11)

ax.set_xlim(xs[0] - 0.3, xs[-1] + 0.3); ax.set_ylim(-0.15, 1.05); ax.axis("off")
ax.set_title(f"{LABELS[STEM]}, {DATASET.capitalize()}  (w = 0.3) — "
             f"posterior inclusion probability per weight\n"
             f"(green = almost always active, red = almost always pruned; "
             f"line weight = certainty)", fontsize=12)
fig.colorbar(mpl.cm.ScalarMappable(norm=norm, cmap=cmap), ax=ax,
             fraction=0.03, pad=0.02, label="P(weight active)")
plt.tight_layout()
plt.show()
# fig.savefig(f"network_inclusion_{DATASET}_{STEM}_w{PIW_VIS:g}.pdf", bbox_inches="tight")


In [ ]:
# --- Keep only weights active > cutoff, re-evaluate on the test set (same sampler) ---
INCL_CUTOFF = 0.50

r          = runs[STEM]
samples    = r["samples"].to(dtype=DTYPE)                     # [S, D]  (last col = log_sigma)
weight_s   = r["weight_samples"]                              # [S, D-1]
keep_mask  = torch.as_tensor(incl_prob > INCL_CUTOFF, dtype=weight_s.dtype)   # [D-1]

weight_masked = weight_s * keep_mask                          # zero the <=cutoff coords in every draw

preds_m = predict_all(bm, weight_masked, X_test)              # [S, N]  (this notebook: bm is 1st arg)
mean_m  = preds_m.mean(0)
noise_m = samples[:, -1].exp()                                # [S]  (log_sigma untouched by the mask)
total_m = (preds_m.std(0) ** 2 + float(noise_m.mean()) ** 2).sqrt()

n_kept = int(keep_mask.sum())
print(f"{r['label']}: kept {n_kept} / {weight_s.shape[1]} weights "
      f"active >{INCL_CUTOFF:.0%} of the time ({n_kept / weight_s.shape[1]:.1%})\n")
print(f"  RMSE    : {compute_rmse(y_test, mean_m, y_std):.4f}   "
      f"(full posterior: {compute_rmse(y_test, r['mean_pred'], y_std):.4f})")
print(f"  NLL     : {compute_nll(y_test, mean_m, total_m, y_std):.4f}   "
      f"(full posterior: {compute_nll(y_test, r['mean_pred'], r['total_std'], y_std):.4f})")
print(f"  CRPS    : {compute_crps(y_test, mean_m, total_m, y_std):.4f}   "
      f"(full posterior: {compute_crps(y_test, r['mean_pred'], r['total_std'], y_std):.4f})")
print(f"  Cov 90% : {compute_coverage(y_test, mean_m, total_m, 0.90):.3f}   "
      f"(full: {compute_coverage(y_test, r['mean_pred'], r['total_std'], 0.90):.3f})")
print(f"  Cov 95% : {compute_coverage(y_test, mean_m, total_m, 0.95):.3f}   "
      f"(full: {compute_coverage(y_test, r['mean_pred'], r['total_std'], 0.95):.3f})")
